# ✂️ Notebook 02: Byte-Pair Encoding (BPE) Tokenizer

Byte-Pair Encoding is a popular subword tokenization technique used in models like GPT and RoBERTa.

**Goals:**
- Understand how BPE works
- Implement a simple version
- Apply it on sample text


In [ ]:
# Sample corpus with repeated patterns to illustrative BPE
corpus = "low lower lowest low lower lowest"

# Split the corpus into tokens (words)
tokens = corpus.split()

# Print the tokens
print(tokens)

['low', 'lower', 'lowest', 'low', 'lower', 'lowest']


## Step 1: Initialize with character-level tokens


In [ ]:
from collections import Counter, defaultdict

# Function to create vocabulary from the tokens
def get_vocab(tokens) :
    vocab = Counter()
    for word in tokens :
        # Convert each word into space-seperated characters + special end-of-word symbol <\w>
        chars = " ".join(list(word)) + r' <\w>'
        vocab[chars] += 1  # Count the frequency of word
    return vocab

# Get vocab from tokens
vocab = get_vocab(tokens)

# Display the initial vocab with frequencies
for word, freq in vocab.items() :
    print(f"{word} : {freq}")

l o w <\w> : 2
l o w e r <\w> : 2
l o w e s t <\w> : 2


## Step 2: Find most frequent pair of symbols


In [ ]:
# Function to count frequency of symbol pairs (bigrams) in the vocab
def get_stats(vocab) :
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()  # Split the word into symbols
        for i in range(len(symbols) - 1) :
            pair = (symbols[i], symbols[i+1])  # Get bigrams
            pairs[pair] += freq  # Add count proportion to word frequency
    return pairs

# Get frequent pairs from the vocab
pairs = get_stats(vocab)

# Top 5 most frequent pairs
print(pairs.most_common(5))

[(('l', 'o'), 6), (('o', 'w'), 6), (('w', 'e'), 4), (('w', '<\\w>'), 2), (('e', 'r'), 2)]


## Step 3: Merge the most frequent pair

In [ ]:
# Function to merge the most frequent pair into a new token
def merge_vocab(pair, vocab) :
    new_vocab = {}
    bigram = ' '.join(pair)  # Convert ('l' , 'o') -> 'l o'
    replacement = ''.join(pair)  # Convert ('l', 'o') -> 'lo'
    for word in vocab :
        # Replace only full bigram occurences with the merged version
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word] # Keep frequency
    return new_vocab

# Merge the top frequent pair once
most_freq = pairs.most_common(1)[0][0] # Get the most frequent pair
vocab = merge_vocab(most_freq, vocab)  # Apply merge
for word in vocab :
    print(word)  # View updated vocab

lo w <\w>
lo w e r <\w>
lo w e s t <\w>


## 🔁 Loop: Perform multiple merges

In [10]:
# Full BPE learner with multiple merge steps
def learn_bpe(tokens, num_merges = 5) :
    vocab = get_vocab(tokens)
    for i in range(num_merges):
        pairs = get_stats(vocab)
        if not pairs :
            break  # Exit if no more pairs to merge
        best = pairs.most_common(1)[0][0]  # Most frequent pair
        vocab = merge_vocab(best, vocab)   # Merge it
        print(f"Merge {i+1} : {best}")
        for word in vocab :
            print(word)
        print('-'*40)
    return vocab

# Run BPE on the token list
learn_bpe(tokens)


Merge 1 : ('l', 'o')
lo w <\w>
lo w e r <\w>
lo w e s t <\w>
----------------------------------------
Merge 2 : ('lo', 'w')
low <\w>
low e r <\w>
low e s t <\w>
----------------------------------------
Merge 3 : ('low', 'e')
low <\w>
lowe r <\w>
lowe s t <\w>
----------------------------------------
Merge 4 : ('low', '<\\w>')
low<\w>
lowe r <\w>
lowe s t <\w>
----------------------------------------
Merge 5 : ('lowe', 'r')
low<\w>
lower <\w>
lowe s t <\w>
----------------------------------------


{'low<\\w>': 2, 'lower <\\w>': 2, 'lowe s t <\\w>': 2}

## ✅ Summary

- BPE helps balance vocabulary size and coverage  
- Handles out-of-vocabulary words better than word-level tokenization  
- Used in GPT, RoBERTa, and other major LLMs
